In [1]:
# ============================================================================
# Kaggle-Optimized: Bottleneck Formation in Deep Networks Under Joint Scarcity
# ============================================================================

# %% Cell 1: Imports and Setup
import os
import sys
import pickle
import hashlib
import time
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
from itertools import product
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import svd as scipy_svd
import pandas as pd

print("Imports successful!")

# %% Cell 2: Kaggle Environment Setup
def setup_kaggle_environment():
    """Setup paths and check Kaggle environment."""
    # Kaggle output directory
    if os.path.exists('/kaggle/working'):
        BASE_DIR = Path('/kaggle/working')
    else:
        BASE_DIR = Path('.')
    
    RESULTS_DIR = BASE_DIR / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
    
    PLOTS_DIR = BASE_DIR / 'plots'
    PLOTS_DIR.mkdir(exist_ok=True)
    
    CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    print(f"✓ Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"✓ Results directory: {RESULTS_DIR}")
    print(f"✓ Plots directory: {PLOTS_DIR}")
    print(f"✓ PyTorch: {torch.__version__}")
    print(f"✓ Lightning: {pl.__version__}")
    
    return RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE

RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE = setup_kaggle_environment()

# %% Cell 3: Time and Memory Management
class ExperimentTimer:
    """Manage experiment time to stay within Kaggle limits."""
    
    def __init__(self, max_hours: float = 11.5):
        self.start_time = time.time()
        self.max_seconds = max_hours * 3600
        
    def elapsed_hours(self) -> float:
        return (time.time() - self.start_time) / 3600
    
    def remaining_hours(self) -> float:
        return self.max_seconds / 3600 - self.elapsed_hours()
    
    def should_stop(self) -> bool:
        return self.elapsed_hours() >= (self.max_seconds / 3600)
    
    def print_status(self):
        print(f"⏱️  Elapsed: {self.elapsed_hours():.2f}h | Remaining: {self.remaining_hours():.2f}h")

def cleanup_memory():
    """Aggressive memory cleanup."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_memory_stats():
    """Get current memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        cached = torch.cuda.memory_reserved() / 1e9
        return f"GPU: {allocated:.2f}GB allocated, {cached:.2f}GB cached"
    return "CPU mode"

# %% Cell 4: Reproducibility
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    pl.seed_everything(seed, workers=True)

# %% Cell 5: Dataset Generation
class GaussianBlobsDataset(Dataset):
    """Synthetic dataset: K classes on hypersphere with Gaussian noise."""
    
    def __init__(self, K: int, d: int, n_per_class: int, seed: int = 42):
        set_seed(seed)
        self.K = K
        self.d = d
        self.n_per_class = n_per_class
        self.N = K * n_per_class
        
        # Generate class centers on hypersphere
        centers = self._generate_centers()
        
        # Generate samples
        X_list, y_list = [], []
        for k in range(K):
            X_k = np.random.randn(n_per_class, d) + centers[k]
            y_k = np.full(n_per_class, k)
            X_list.append(X_k)
            y_list.append(y_k)
        
        self.X = torch.FloatTensor(np.vstack(X_list))
        self.y = torch.LongTensor(np.hstack(y_list))
        
    def _generate_centers(self) -> np.ndarray:
        """Generate K orthogonal class centers on sqrt(d) * hypersphere."""
        A = np.random.randn(self.d, self.K)
        Q, _ = np.linalg.qr(A)
        centers = np.sqrt(self.d) * Q.T[:self.K]
        return centers
    
    def __len__(self):
        return self.N
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_data_loaders(K: int, d: int, n_per_class: int, 
                        batch_size: int, seed: int = 42):
    """Create train/val loaders with 90/10 split."""
    dataset = GaussianBlobsDataset(K, d, n_per_class, seed)
    N = len(dataset)
    n_val = max(1, int(0.1 * N))
    n_train = N - n_val
    
    train_ds, val_ds = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(seed)
    )
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                              num_workers=0, persistent_workers=False, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                           num_workers=0, persistent_workers=False, pin_memory=False)
    
    return train_loader, val_loader, dataset

# %% Cell 6: Network Architecture
class BottleneckMLP(nn.Module):
    """Deep MLP with configurable bottleneck architecture."""
    
    def __init__(self, d0: int, K: int, L_hidden: int, d_b: int,
                 width_pattern: str = 'constant_narrow', use_relu: bool = True):
        super().__init__()
        self.d0 = d0
        self.K = K
        self.L_hidden = L_hidden
        self.d_b = d_b
        self.width_pattern = width_pattern
        self.use_relu = use_relu
        
        widths = self._build_widths()
        self.widths = widths
        
        layers = []
        for i in range(len(widths) - 1):
            layers.append(nn.Linear(widths[i], widths[i+1]))
            if i < len(widths) - 2:
                if use_relu:
                    layers.append(nn.ReLU())
        
        self.network = nn.Sequential(*layers)
        
    def _build_widths(self) -> List[int]:
        """Build layer width schedule."""
        if self.width_pattern == 'constant_narrow':
            widths = [self.d0] + [self.d_b] * self.L_hidden + [self.K]
        elif self.width_pattern == 'wide_narrow_wide':
            if self.L_hidden < 3:
                widths = [self.d0, self.d_b, self.K]
            else:
                n_wide = (self.L_hidden - 1) // 2
                widths = ([self.d0] + [256] * n_wide + [self.d_b] + 
                         [256] * (self.L_hidden - n_wide - 1) + [self.K])
        elif self.width_pattern == 'constant_wide':
            widths = [self.d0] + [1024] * self.L_hidden + [self.K]
        else:
            raise ValueError(f"Unknown width pattern: {self.width_pattern}")
        
        return widths
    
    def forward(self, x):
        return self.network(x)
    
    def get_layer_activations(self, x: torch.Tensor) -> List[torch.Tensor]:
        """Extract activations at each layer."""
        activations = [x]
        h = x
        for layer in self.network:
            h = layer(h)
            if isinstance(layer, (nn.Linear, nn.ReLU)):
                activations.append(h.detach())
        return activations

# %% Cell 7: Metrics Computation
class MetricsComputer:
    """Compute representational metrics for bottleneck analysis."""
    
    @staticmethod
    def stable_rank(H: torch.Tensor) -> float:
        """Stable rank: ||H||_F^2 / ||H||_2^2"""
        if H.shape[0] < 2:
            return float(H.shape[1])
        frob_norm = torch.norm(H, p='fro').item() ** 2
        spec_norm = torch.linalg.matrix_norm(H, ord=2).item() ** 2
        return frob_norm / (spec_norm + 1e-10)
    
    @staticmethod
    def nc1_metric(H: torch.Tensor, y: torch.Tensor, K: int) -> float:
        """NC1: Tr(Σ_W) / Tr(Σ_T)."""
        H_np = H.cpu().numpy()
        y_np = y.cpu().numpy()
        
        mu_G = H_np.mean(axis=0)
        
        Sigma_W = 0
        Sigma_T = 0
        for k in range(K):
            H_k = H_np[y_np == k]
            if len(H_k) == 0:
                continue
            mu_k = H_k.mean(axis=0)
            H_k_centered = H_k - mu_k
            Sigma_W += H_k_centered.T @ H_k_centered
            n_k = len(H_k)
            mu_k_diff = mu_k - mu_G
            Sigma_T += n_k * np.outer(mu_k_diff, mu_k_diff)
        
        Sigma_T += Sigma_W
        
        tr_W = np.trace(Sigma_W)
        tr_T = np.trace(Sigma_T)
        
        return float(tr_W / (tr_T + 1e-10))
    
    @staticmethod
    def singular_values(H: torch.Tensor, k: int = 10) -> np.ndarray:
        """Top-k singular values."""
        if H.shape[0] < k:
            k = H.shape[0]
        U, S, V = torch.svd_lowrank(H, q=min(k, min(H.shape)))
        return S.cpu().numpy()
    
    @staticmethod
    def compute_all_metrics(model: nn.Module, loader: DataLoader, 
                           K: int, device: str) -> Dict:
        """Compute all metrics for each layer."""
        model.eval()
        model = model.to(device)
        
        all_X, all_y = [], []
        for X, y in loader:
            all_X.append(X)
            all_y.append(y)
        X_full = torch.cat(all_X, dim=0).to(device)
        y_full = torch.cat(all_y, dim=0).to(device)
        
        with torch.no_grad():
            activations = model.get_layer_activations(X_full)
        
        metrics = {}
        L = len(activations) - 1
        
        for ell in range(1, L + 1):
            H = activations[ell]
            if len(H.shape) == 1:
                H = H.unsqueeze(0)
            
            if ell == L or H.shape[1] == K:
                continue
            
            srank = MetricsComputer.stable_rank(H)
            nc1 = MetricsComputer.nc1_metric(H, y_full, K)
            svals = MetricsComputer.singular_values(H, k=10)
            
            metrics[f'layer_{ell}'] = {
                'stable_rank': srank,
                'nc1': nc1,
                'singular_values': svals,
                'width': H.shape[1]
            }
        
        # Cleanup
        del X_full, y_full, activations
        cleanup_memory()
        
        return metrics

# %% Cell 8: Experiment Configuration
@dataclass
class ExperimentConfig:
    """Configuration for a single experiment."""
    K: int = 10
    d0: int = 1000
    n_per_class: int = 10
    d_b: int = 64
    L_hidden: int = 5
    width_pattern: str = 'constant_narrow'
    wd: float = 0.0
    lr: float = 0.1
    max_iters: int = 800
    batch_size: int = -1
    seed: int = 42
    use_relu: bool = True
    
    def to_hash(self) -> str:
        """Generate unique hash for config (excluding seed)."""
        config_dict = asdict(self)
        config_dict.pop('seed')
        config_str = str(sorted(config_dict.items()))
        return hashlib.md5(config_str.encode()).hexdigest()[:8]

# %% Cell 9: Lightning Module
class BottleneckLightningModule(pl.LightningModule):
    """PyTorch Lightning module for training and metrics."""
    
    def __init__(self, config: ExperimentConfig):
        super().__init__()
        self.save_hyperparameters()
        self.config = config
        
        self.model = BottleneckMLP(
            d0=config.d0,
            K=config.K,
            L_hidden=config.L_hidden,
            d_b=config.d_b,
            width_pattern=config.width_pattern,
            use_relu=config.use_relu
        )
        
        self.criterion = nn.CrossEntropyLoss()
        
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.parameters(),
            lr=self.config.lr,
            momentum=0.9,
            weight_decay=self.config.wd
        )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.config.max_iters
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'step'
            }
        }

# %% Cell 10: Training Function
def train_single_run(config: ExperimentConfig, timer: ExperimentTimer) -> Optional[Dict]:
    """Train a single configuration and return metrics."""
    if timer.should_stop():
        print("⏱️  Time limit reached, skipping this run")
        return None
    
    set_seed(config.seed)
    
    try:
        # Create data
        N = config.K * config.n_per_class
        batch_size = N if config.batch_size == -1 else config.batch_size
        train_loader, val_loader, dataset = create_data_loaders(
            config.K, config.d0, config.n_per_class, batch_size, config.seed
        )
        
        # Model
        model = BottleneckLightningModule(config)
        
        # Callbacks
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=50,
            mode='min'
        )
        
        # Trainer
        trainer = pl.Trainer(
            max_steps=config.max_iters,
            logger=False,
            callbacks=[early_stop],
            enable_progress_bar=False,
            enable_checkpointing=False,
            accelerator='auto',
            devices=1,
            deterministic=True
        )
        
        # Train
        trainer.fit(model, train_loader, val_loader)
        
        # Compute final metrics
        device = next(model.parameters()).device
        model.eval()
        metrics = MetricsComputer.compute_all_metrics(
            model.model, train_loader, config.K, str(device)
        )
        
        result = {
            'config': asdict(config),
            'metrics': metrics,
            'final_train_acc': trainer.callback_metrics.get('train_acc', torch.tensor(0.0)).item(),
            'final_val_loss': trainer.callback_metrics.get('val_loss', torch.tensor(float('inf'))).item()
        }
        
        # Cleanup
        del model, trainer, train_loader, val_loader, dataset
        cleanup_memory()
        
        return result
        
    except Exception as e:
        print(f"❌ Run failed: {e}")
        cleanup_memory()
        return None

# %% Cell 11: Checkpoint Management
def save_checkpoint(results: List[Dict], checkpoint_path: Path):
    """Save intermediate results."""
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"💾 Checkpoint saved: {len(results)} results")

def load_checkpoint(checkpoint_path: Path) -> List[Dict]:
    """Load checkpoint if exists."""
    if checkpoint_path.exists():
        with open(checkpoint_path, 'rb') as f:
            results = pickle.load(f)
        print(f"📂 Loaded checkpoint: {len(results)} results")
        return results
    return []

# %% Cell 12: Sweep Function
def run_sweep(sweep_params: Dict, n_seeds: int, timer: ExperimentTimer, 
              checkpoint_path: Path) -> List[Dict]:
    """Run full hyperparameter sweep with time management."""
    
    # Load existing results
    all_results = load_checkpoint(checkpoint_path)
    
    # Generate all configurations
    keys = list(sweep_params.keys())
    values = [sweep_params[k] for k in keys]
    
    completed_configs = set()
    for res in all_results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'], 
               cfg['width_pattern'], cfg['wd'], cfg['seed'])
        completed_configs.add(key)
    
    config_count = 0
    total_configs = np.prod([len(v) for v in values]) * n_seeds
    
    for combo in product(*values):
        config_dict = dict(zip(keys, combo))
        
        for seed in range(n_seeds):
            config = ExperimentConfig(**config_dict, seed=seed)
            
            # Skip if already completed
            key = (config.n_per_class, config.d_b, config.L_hidden,
                   config.width_pattern, config.wd, seed)
            if key in completed_configs:
                continue
            
            config_count += 1
            
            # Check time limit
            if timer.should_stop():
                print(f"\n⏱️  Time limit reached after {config_count} runs")
                print(f"📊 Completed: {len(all_results)}/{total_configs} runs")
                break
            
            print(f"\n[{len(all_results)+1}/{total_configs}] n={config.n_per_class}, "
                  f"d_b={config.d_b}, L={config.L_hidden}, "
                  f"pattern={config.width_pattern[:10]}, wd={config.wd}, seed={seed}")
            timer.print_status()
            print(f"🔧 {get_memory_stats()}")
            
            try:
                result = train_single_run(config, timer)
                if result is not None:
                    all_results.append(result)
                    
                    # Save checkpoint every 5 runs
                    if len(all_results) % 5 == 0:
                        save_checkpoint(all_results, checkpoint_path)
                        
            except Exception as e:
                print(f"❌ Run failed: {e}")
                cleanup_memory()
                continue
        
        if timer.should_stop():
            break
    
    # Final save
    save_checkpoint(all_results, checkpoint_path)
    
    return all_results

# %% Cell 13: Visualization Functions
def plot_rank_evolution(results: List[Dict], save_path: Path):
    """Plot stable rank evolution across layers."""
    if len(results) == 0:
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['width_pattern'])
        if key not in config_groups:
            config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, pattern = key
        
        layer_ranks = {}
        for res in group:
            metrics = res['metrics']
            for layer_name, layer_metrics in metrics.items():
                if layer_name.startswith('layer_'):
                    layer_idx = int(layer_name.split('_')[1])
                    if layer_idx not in layer_ranks:
                        layer_ranks[layer_idx] = []
                    layer_ranks[layer_idx].append(layer_metrics['stable_rank'])
        
        layers = sorted(layer_ranks.keys())
        means = [np.mean(layer_ranks[l]) for l in layers]
        stds = [np.std(layer_ranks[l]) for l in layers]
        
        ax.plot(layers, means, 'o-', linewidth=2, markersize=8)
        ax.fill_between(layers, 
                        np.array(means) - np.array(stds),
                        np.array(means) + np.array(stds),
                        alpha=0.3)
        
        ax.set_xlabel('Layer ℓ', fontsize=12)
        ax.set_ylabel('Stable Rank', fontsize=12)
        ax.set_title(f'n={n}, d_b={d_b}, {pattern}', fontsize=11)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_singular_spectra(results: List[Dict], save_path: Path):
    """Plot singular value spectra."""
    if len(results) == 0:
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'])
        if key not in config_groups:
            config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, L = key
        
        for res in group[:1]:
            metrics = res['metrics']
            middle_layer = L // 2 + 1
            layer_key = f'layer_{middle_layer}'
            
            if layer_key in metrics:
                svals = metrics[layer_key]['singular_values']
                ax.loglog(range(1, len(svals)+1), svals, 'o-', linewidth=2)
        
        ax.set_xlabel('Singular Value Index', fontsize=12)
        ax.set_ylabel('Singular Value', fontsize=12)
        ax.set_title(f'n={n}, d_b={d_b}, L={L}', fontsize=11)
        ax.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_scaling_law(results: List[Dict], save_path: Path):
    """Plot scaling law analysis."""
    if len(results) < 10:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                min_rank = min(min_rank, rank)
        
        if min_rank < float('inf'):
            data.append({
                'n': cfg['n_per_class'],
                'K': cfg['K'],
                'd_b': cfg['d_b'],
                'nK': cfg['n_per_class'] * cfg['K'],
                'r_min': min_rank
            })
    
    df = pd.DataFrame(data)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].loglog(df['nK'], df['r_min'], 'o', alpha=0.5)
    axes[0].set_xlabel('nK (total samples)', fontsize=12)
    axes[0].set_ylabel('Min Stable Rank', fontsize=12)
    axes[0].set_title('Scaling with Data', fontsize=13)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].loglog(df['d_b'], df['r_min'], 'o', alpha=0.5)
    axes[1].set_xlabel('Bottleneck Width d_b', fontsize=12)
    axes[1].set_ylabel('Min Stable Rank', fontsize=12)
    axes[1].set_title('Scaling with Width', fontsize=13)
    axes[1].grid(True, alpha=0.3)
    
    axes[2].hist(df['r_min'], bins=20, alpha=0.7, edgecolor='black')
    axes[2].set_xlabel('Min Stable Rank', fontsize=12)
    axes[2].set_ylabel('Frequency', fontsize=12)
    axes[2].set_title('Distribution', fontsize=13)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def generate_summary_table(results: List[Dict], save_path: Path):
    """Generate summary statistics table."""
    if len(results) == 0:
        return
    
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        
        ranks = {}
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                layer_idx = int(layer_name.split('_')[1])
                ranks[layer_idx] = layer_metrics['stable_rank']
        
        if len(ranks) > 0:
            min_rank = min(ranks.values())
            data.append({
                'n': cfg['n_per_class'],
                'd_b': cfg['d_b'],
                'L': cfg['L_hidden'],
                'pattern': cfg['width_pattern'][:10],
                'wd': cfg['wd'],
                'r_min': min_rank,
                'acc': res['final_train_acc']
            })
    
    df = pd.DataFrame(data)
    summary = df.groupby(['n', 'd_b', 'L', 'pattern', 'wd']).agg({
        'r_min': ['mean', 'std'],
        'acc': 'mean'
    }).round(3)
    
    summary.to_csv(save_path)
    print(f"✓ Saved: {save_path.name}")
    print("\nSummary Statistics:")
    print(summary.head(10))

# %% Cell 14: Quick Test Run
print("\n" + "="*80)
print("🧪 RUNNING QUICK TEST (should complete in <1 minute)")
print("="*80)

test_timer = ExperimentTimer(max_hours=0.1)
test_config = ExperimentConfig(
    n_per_class=5,
    d_b=32,
    L_hidden=3,
    max_iters=100,
    width_pattern='constant_narrow',
    wd=0.0,
    seed=42
)

print(f"Testing with: n=5, d_b=32, L=3, max_iters=100")
test_result = train_single_run(test_config, test_timer)

if test_result is not None:
    print("✅ TEST PASSED! Proceeding to full sweep...")
    cleanup_memory()
else:
    print("❌ TEST FAILED! Check configuration and try again.")
    sys.exit(1)

# %% Cell 15: Full Sweep Configuration
print("\n" + "="*80)
print("🚀 FULL SWEEP CONFIGURATION")
print("="*80)

sweep_params = {
    'n_per_class': [5, 10, 20],
    'd_b': [32, 64, 128],
    'L_hidden': [5, 10],
    'width_pattern': ['constant_narrow', 'wide_narrow_wide'],
    'wd': [0, 1e-4, 1e-3],
    'K': [10]
}

n_seeds = 5  # Reduced for Kaggle time limits
max_iters = 800

total_configs = np.prod([len(v) for v in sweep_params.values()])
total_runs = total_configs * n_seeds

print(f"Unique configurations: {total_configs}")
print(f"Seeds per config: {n_seeds}")
print(f"Total runs: {total_runs}")
print(f"Max iterations per run: {max_iters}")
print(f"Estimated time: ~{total_runs * 2 / 60:.1f} hours (rough estimate)")
print("="*80)

# Update config defaults
ExperimentConfig.__dataclass_fields__['max_iters'].default = max_iters

# %% Cell 16: Run Full Sweep
print("\n🏃 Starting full sweep...")
main_timer = ExperimentTimer(max_hours=11.5)
checkpoint_path = CHECKPOINT_DIR / 'sweep_checkpoint.pkl'

all_results = run_sweep(
    sweep_params=sweep_params,
    n_seeds=n_seeds,
    timer=main_timer,
    checkpoint_path=checkpoint_path
)

print(f"\n✅ Sweep complete! {len(all_results)} results collected.")
print(f"⏱️  Total time: {main_timer.elapsed_hours():.2f} hours")

# %% Cell 17: Generate All Plots
print("\n" + "="*80)
print("📊 GENERATING VISUALIZATIONS")
print("="*80)

if len(all_results) > 0:
    print(f"Processing {len(all_results)} results...")
    
    plot_rank_evolution(all_results, PLOTS_DIR / 'rank_evolution.png')
    plot_singular_spectra(all_results, PLOTS_DIR / 'singular_spectra.png')
    plot_scaling_law(all_results, PLOTS_DIR / 'scaling_law.png')
    generate_summary_table(all_results, RESULTS_DIR / 'summary_stats.csv')
    
    # Save final results
    with open(RESULTS_DIR / 'all_results.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    print(f"\n✓ Saved all results: {RESULTS_DIR / 'all_results.pkl'}")
    
    # Export to CSV
    rows = []
    for res in all_results:
        cfg = res['config']
        base_row = {
            'n_per_class': cfg['n_per_class'],
            'd_b': cfg['d_b'],
            'L_hidden': cfg['L_hidden'],
            'width_pattern': cfg['width_pattern'],
            'wd': cfg['wd'],
            'seed': cfg['seed'],
            'final_train_acc': res.get('final_train_acc', 0),
            'final_val_loss': res.get('final_val_loss', float('inf'))
        }
        
        metrics = res['metrics']
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                row = base_row.copy()
                row['layer'] = layer_name
                row['stable_rank'] = layer_metrics['stable_rank']
                row['nc1'] = layer_metrics['nc1']
                row['width'] = layer_metrics['width']
                rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / 'detailed_results.csv', index=False)
    print(f"✓ Saved detailed CSV: {RESULTS_DIR / 'detailed_results.csv'}")
    
else:
    print("⚠️  No results to visualize")

# %% Cell 18: Final Summary
print("\n" + "="*80)
print("📈 EXPERIMENT SUMMARY")
print("="*80)

if len(all_results) > 0:
    print(f"Total successful runs: {len(all_results)}")
    print(f"Total configurations tested: {len(set([(r['config']['n_per_class'], r['config']['d_b'], r['config']['L_hidden']) for r in all_results]))}")
    
    # Calculate key statistics
    min_ranks = []
    final_accs = []
    
    for res in all_results:
        metrics = res['metrics']
        ranks = [m['stable_rank'] for m in metrics.values() if 'stable_rank' in m]
        if ranks:
            min_ranks.append(min(ranks))
        final_accs.append(res.get('final_train_acc', 0))
    
    if min_ranks:
        print(f"\nBottleneck Characteristics:")
        print(f"  Mean minimum rank: {np.mean(min_ranks):.2f} ± {np.std(min_ranks):.2f}")
        print(f"  Median minimum rank: {np.median(min_ranks):.2f}")
    
    if final_accs:
        print(f"\nTraining Performance:")
        print(f"  Mean final accuracy: {np.mean(final_accs):.3f}")
        print(f"  Runs with acc > 0.9: {sum(np.array(final_accs) > 0.9)}/{len(final_accs)}")
    
    print(f"\n📁 Output Files:")
    print(f"  Results: {RESULTS_DIR}/")
    print(f"  Plots: {PLOTS_DIR}/")
    print(f"  Checkpoint: {checkpoint_path}")
    
else:
    print("⚠️  No results generated")

print("="*80)
print("\n✅ EXPERIMENT COMPLETE!")
print(f"⏱️  Total runtime: {main_timer.elapsed_hours():.2f} hours")
print("="*80)

Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Imports successful!
✓ Device: cuda
✓ GPU: Tesla P100-PCIE-16GB
✓ GPU Memory: 17.06 GB
✓ Results directory: /kaggle/working/results
✓ Plots directory: /kaggle/working/plots
✓ PyTorch: 2.8.0+cu126
✓ Lightning: 2.6.0

🧪 RUNNING QUICK TEST (should complete in <1 minute)
Testing with: n=5, d_b=32, L=3, max_iters=100


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BottleneckMLP    │ 34.5 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 34.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 34.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_steps=100` reached.
Seed set to 0
Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


... (2,301 outputs left out)

💾 Checkpoint saved: 375 results

[376/540] n=20, d_b=128, L=10, pattern=wide_narro, wd=0.001, seed=4
⏱️  Elapsed: 0.17h | Remaining: 11.33h
🔧 GPU: 0.00GB allocated, 0.00GB cached


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BottleneckMLP    │  785 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 785 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 785 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

💾 Checkpoint saved: 376 results

✅ Sweep complete! 376 results collected.
⏱️  Total time: 0.17 hours

📊 GENERATING VISUALIZATIONS
Processing 376 results...
✓ Saved: rank_evolution.png
✓ Saved: singular_spectra.png
✓ Saved: scaling_law.png
✓ Saved: summary_stats.csv

Summary Statistics:
                            r_min           acc
                             mean    std   mean
n d_b L  pattern    wd                         
5 32  5  constant_n 0.0000  0.211  0.471  0.133
                    0.0001  0.000  0.000  0.111
                    0.0010  0.269  0.539  0.150
         wide_narro 0.0001  0.000  0.000  0.111
                    0.0010  0.500  0.707  0.167
      10 constant_n 0.0000  1.000  0.000  0.107
                    0.0001  1.000  0.000  0.111
                    0.0010  1.000  0.000  0.111
         wide_narro 0.0000  1.000  0.000  0.124
                    0.0001  1.000  0.000  0.120

✓ Saved all results: /kaggle/working/results/all_results.pkl
✓ Saved detailed CSV: /kagg